# 08 -- Sprawnosc generatora i straty elektryczne

Ten notebook przedstawia modele strat elektrycznych w MEW:

1. **Generator** -- model strat miedzio-zelaznych eta_g(P/P_n)
2. **Transformator** -- straty jałowe i obciazeniowe (opcjonalny)
3. **Potrzeby wlasne** -- zuzycie energii na cele wlasne elektrowni
4. **Sprawnosc calkowita elektryczna** -- zlozenie modeli

Modele zaimplementowane w module `src/generator.py`.

---

## Modul `src/generator.py`

**Prompt do LLM tworzacy ten modul:**
> *"Stworz modul src/generator.py do modelowania strat elektrycznych MEW.
> (1) generator_efficiency(P_ratio, k_cu, k_fe, k_mech) -- model strat generatora:
> straty miedziane (proporcjonalne do P^2), zelazne (stale), mechaniczne (stale).
> (2) transformer_efficiency(P_ratio, k_cu, k_fe) -- analogiczny model transformatora.
> (3) auxiliary_power(P_rated, aux_fraction, aux_fixed) -- potrzeby wlasne.
> (4) electrical_efficiency -- zlozenie generatora i transformatora."*

**Funkcje w module:**
- `generator_efficiency(P_ratio, k_cu, k_fe, k_mech)` -- sprawnosc generatora
- `generator_efficiency_peak(k_cu, k_fe, k_mech)` -- punkt optymalny
- `transformer_efficiency(P_ratio, k_cu, k_fe)` -- sprawnosc transformatora
- `auxiliary_power(P_rated, aux_fraction, aux_fixed)` -- potrzeby wlasne
- `electrical_efficiency(...)` -- sprawnosc calkowita elektryczna

## Konfiguracja

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.generator import (
    generator_efficiency,
    generator_efficiency_peak,
    transformer_efficiency,
    auxiliary_power,
    electrical_efficiency,
)

P_ratio = np.linspace(0.01, 1.2, 300)
print('Modul zaladowany.')

---
## Krok 1: Model sprawnosci generatora

Straty w generatorze synchronicznym skladaja sie z trzech komponentow:

$$\eta_g = \frac{P}{P + \Delta P_{Cu} + \Delta P_{Fe} + \Delta P_{mech}}$$

gdzie (wyrzone jako ulamki mocy znamionowej $P_n$):

| Skladnik | Wzor | Charakter | Typowe wartosci |
|----------|------|-----------|----------------|
| $\Delta P_{Cu}$ | $k_{Cu} \cdot (P/P_n)^2$ | Rosnie z obciazeniem ($\sim I^2 R$) | 1.5--3.5% |
| $\Delta P_{Fe}$ | $k_{Fe}$ | Stale (histereza + prady wirowe) | 0.5--1.5% |
| $\Delta P_{mech}$ | $k_{mech}$ | Stale (lozyska, wentylacja) | 0.3--0.8% |

**Sprawnosc maksymalna** wystepuje gdy straty zmienne = straty stale:

$$P_{opt}/P_n = \sqrt{\frac{k_{Fe} + k_{mech}}{k_{Cu}}}$$

**Prompt do LLM:**
> *"Narysuj krzywe sprawnosci generatora dla 3 zestawow parametrow:
> (1) maly generator (k_cu=0.035, k_fe=0.012, k_mech=0.008),
> (2) sredni (domyslne), (3) duzy/nowoczesny (k_cu=0.015, k_fe=0.005, k_mech=0.003).
> Zaznacz punkt optymalny dla kazdego."*

**Uzyte funkcje:** `generator_efficiency()`, `generator_efficiency_peak()`

In [ ]:
configs = [
    ('Maly generator', 0.035, 0.012, 0.008, 'firebrick'),
    ('Sredni (domyslny)', 0.025, 0.010, 0.005, 'royalblue'),
    ('Duzy/nowoczesny', 0.015, 0.005, 0.003, 'green'),
]

fig = go.Figure()
for label, k_cu, k_fe, k_mech, color in configs:
    eta = generator_efficiency(P_ratio, k_cu, k_fe, k_mech)
    P_opt, eta_max = generator_efficiency_peak(k_cu, k_fe, k_mech)
    eta_full = float(generator_efficiency(1.0, k_cu, k_fe, k_mech))

    fig.add_trace(go.Scatter(
        x=P_ratio * 100, y=eta * 100,
        mode='lines', name=label,
        line=dict(color=color, width=2.5),
    ))
    # Mark peak
    fig.add_trace(go.Scatter(
        x=[P_opt * 100], y=[eta_max * 100],
        mode='markers', showlegend=False,
        marker=dict(color=color, size=10, symbol='star'),
    ))
    print(f'{label}: eta_max={eta_max:.4f} przy P/Pn={P_opt:.1%}, eta(100%)={eta_full:.4f}')

fig.update_layout(
    title='Sprawnosc generatora -- eta_g(P/P_rated)',
    xaxis_title='Obciazenie P/P_n [%]',
    yaxis_title='Sprawnosc eta_g [%]',
    yaxis_range=[80, 100],
    height=500, hovermode='x unified',
)
fig.show()

### Rozklad strat generatora

Zobaczmy jak poszczegolne skladniki strat zmieniaja sie z obciazeniem:

In [ ]:
k_cu, k_fe, k_mech = 0.025, 0.010, 0.005

loss_cu = k_cu * P_ratio**2 * 100    # %
loss_fe = np.full_like(P_ratio, k_fe * 100)
loss_mech = np.full_like(P_ratio, k_mech * 100)

fig = go.Figure()
fig.add_trace(go.Scatter(x=P_ratio*100, y=loss_cu, mode='lines',
    name='Straty Cu (obciazeniowe)', fill='tozeroy',
    line=dict(color='firebrick')))
fig.add_trace(go.Scatter(x=P_ratio*100, y=loss_cu + loss_fe, mode='lines',
    name='Straty Fe (jalowe)', fill='tonexty',
    line=dict(color='royalblue')))
fig.add_trace(go.Scatter(x=P_ratio*100, y=loss_cu + loss_fe + loss_mech, mode='lines',
    name='Straty mech.', fill='tonexty',
    line=dict(color='green')))

fig.update_layout(
    title='Rozklad strat generatora (k_cu=0.025, k_fe=0.010, k_mech=0.005)',
    xaxis_title='Obciazenie P/P_n [%]',
    yaxis_title='Straty [% P_n]',
    height=400, hovermode='x unified',
)
fig.show()

---
## Krok 2: Transformator (opcjonalny)

Transformator podwyzszajacy napiecie ma analogiczna strukture strat:

$$\eta_{tr} = \frac{P}{P + k_{Cu,tr} \cdot (P/P_n)^2 + k_{Fe,tr}}$$

Straty transformatora sa mniejsze niz generatora (brak czesci ruchomych):
- $k_{Cu,tr}$ = 0.8--1.5%
- $k_{Fe,tr}$ = 0.3--0.8%

Transformator jest **opcjonalny** w modelu -- nie kazda MEW go wymaga
(np. przy niskim napieciu i krotkim przesyle).

In [ ]:
eta_g = generator_efficiency(P_ratio)
eta_t = transformer_efficiency(P_ratio)
eta_both = electrical_efficiency(P_ratio, include_trafo=True)
eta_no_trafo = electrical_efficiency(P_ratio, include_trafo=False)

fig = go.Figure()
fig.add_trace(go.Scatter(x=P_ratio*100, y=eta_g*100, mode='lines',
    name='Generator', line=dict(color='royalblue', width=2)))
fig.add_trace(go.Scatter(x=P_ratio*100, y=eta_t*100, mode='lines',
    name='Transformator', line=dict(color='green', width=2)))
fig.add_trace(go.Scatter(x=P_ratio*100, y=eta_both*100, mode='lines',
    name='Generator + Transformator', line=dict(color='firebrick', width=2.5)))

fig.update_layout(
    title='Sprawnosc elektryczna -- generator i transformator',
    xaxis_title='Obciazenie P/P_n [%]',
    yaxis_title='Sprawnosc [%]',
    yaxis_range=[80, 100],
    height=450, hovermode='x unified',
)
fig.show()

print(f'Przy pelnym obciazeniu:')
print(f'  Generator:     {float(generator_efficiency(1.0)):.1%}')
print(f'  Transformator: {float(transformer_efficiency(1.0)):.1%}')
print(f'  Lacznie:       {float(electrical_efficiency(1.0)):.1%}')

---
## Krok 3: Potrzeby wlasne

Czesc wyprodukowanej energii zuzywa sama elektrownia:

$$P_{aux} = P_{aux,stale} + k_{aux} \cdot P_n$$

Obejmuje: uklady sterowania, chlodzenie, oswietlenie, pompy olejowe, suwnica, itp.

Typowe wartosci:
- $P_{aux,stale}$ = 2--10 kW
- $k_{aux}$ = 1--3% mocy znamionowej

In [ ]:
P_rated_range = np.linspace(50, 2000, 100)
P_aux = [auxiliary_power(P) for P in P_rated_range]
P_aux_pct = [auxiliary_power(P) / P * 100 for P in P_rated_range]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Potrzeby wlasne [kW]', 'Potrzeby wlasne [% P_n]'])

fig.add_trace(go.Scatter(x=P_rated_range, y=P_aux, mode='lines',
    line=dict(color='royalblue', width=2), name='P_aux'), row=1, col=1)
fig.add_trace(go.Scatter(x=P_rated_range, y=P_aux_pct, mode='lines',
    line=dict(color='firebrick', width=2), name='P_aux [%]'), row=1, col=2)

fig.update_xaxes(title_text='Moc znamionowa P_n [kW]', row=1, col=1)
fig.update_xaxes(title_text='Moc znamionowa P_n [kW]', row=1, col=2)
fig.update_yaxes(title_text='P_aux [kW]', row=1, col=1)
fig.update_yaxes(title_text='P_aux [%]', row=1, col=2)
fig.update_layout(height=400, showlegend=False,
    title='Potrzeby wlasne MEW')
fig.show()

# Przyklad
P_n = 500
print(f'MEW {P_n} kW: potrzeby wlasne = {auxiliary_power(P_n):.1f} kW ({auxiliary_power(P_n)/P_n*100:.1f}%)')

---
## Krok 4: Porownanie z modelem stalym

Notebook 05 uzywal stalej sprawnosci $\eta_{total} = 0.848$.
Zobaczmy jak wyglada roznica miedzy modelem stalym a zmiennym.

In [ ]:
eta_const = 0.96  # stala sprawnosc generatora z nb05
eta_var = generator_efficiency(P_ratio)

fig = go.Figure()
fig.add_trace(go.Scatter(x=P_ratio*100, y=np.full_like(P_ratio, eta_const)*100,
    mode='lines', name='Stala (nb05)', line=dict(color='gray', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=P_ratio*100, y=eta_var*100,
    mode='lines', name='Zmienna (model)', line=dict(color='royalblue', width=2.5)))

fig.update_layout(
    title='Stala vs zmienna sprawnosc generatora',
    xaxis_title='Obciazenie P/P_n [%]',
    yaxis_title='Sprawnosc eta_g [%]',
    yaxis_range=[80, 100],
    height=400, hovermode='x unified',
)
fig.show()

print('Model staly zawyza sprawnosc przy niskich obciazeniach')
print('i lekko zanizy przy optymalnym obciazeniu (~77%).')

---
## Podsumowanie

| Model | Opis | Funkcja |
|-------|------|---------|
| Generator | eta_g(P/Pn) = P/(P + k_cu*P^2 + k_fe + k_mech) | `generator_efficiency()` |
| Transformator | eta_tr(P/Pn) = P/(P + k_cu*P^2 + k_fe) | `transformer_efficiency()` |
| Potrzeby wlasne | P_aux = P_stale + k*Pn | `auxiliary_power()` |
| Calkowita | eta_g * eta_tr | `electrical_efficiency()` |

**Kluczowe wnioski:**
- Sprawnosc generatora spada ponizej ~30% obciazenia
- Maksimum sprawnosci nie jest przy 100%, lecz przy ~75-80%
- Transformator dodaje dodatkowe ~1.5% strat
- Model staly (nb05) nie oddaje zachowania przy czesciowym obciazeniu

**Dalej:** notebook 09 -- model kosztow